# V5: matched CNN sentence-label shuffle control
Run the confirmed raw-input CNN at LR=0.0001 on the same participants, seeds 45/62, validation splits, and checkpoint rule, but assign a balanced random label mapping to sentence IDs. Each mapping is shared across all participants and matches the preceding logistic-control mapping.
This is 24 new fits, roughly one hour at the previous GPU speed. Code/data/runtime/training settings are checked against the real-label CNN before training. It uses the same 100-epoch limit and early-stopping rule; the target mapping is what changes.
Good performance on shuffled sentence labels would suggest that transferable stimulus fingerprints can explain some decoding. It would not by itself prove a coding bug. Two mappings are a diagnostic rather than a formal permutation significance test.

## 1. Setup
Select a GPU runtime. Code is fixed at tag `v5-cnn-shuffle-control-v1`.

In [ ]:
import os, sys, subprocess, json
from google.colab import drive
drive.mount('/content/drive')
repo = '/content/teco-v5-cnn-control'
run_ref = 'v5-cnn-shuffle-control-v1'
if not os.path.isdir(repo):
    subprocess.run(['git', 'clone', '-q', '--branch', run_ref, '--single-branch',
                    'https://github.com/parmisbathayan/reproduce-teco-results.git', repo], check=True)
os.chdir(repo)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert commit == subprocess.check_output(['git', 'rev-parse', run_ref + '^{commit}'], text=True).strip()
assert not subprocess.check_output(['git', 'status', '--porcelain'], text=True).strip(), 'Use a clean fixed revision.'
print('Ready. Git commit:', commit, flush=True)


## 2. Run or resume the matched control
Every fit prints START and a saved-result line. Completed fits are skipped after a reconnect.

In [ ]:
base = '/content/drive/MyDrive/Thesis/Results/reproduce_yaser_codes/eegonly'
out_dir = os.path.join(base, 'v5_cnn_shuffle_control_45_62')
reference_dir = os.path.join(base, 'v4_low_lr_confirmation_45_62')
command = [sys.executable, '-u', '-B', 'src/v5_baselines.py',
           '--trt-dir', '/content/drive/MyDrive/Thesis/Data/TRT_Total',
           '--labels-csv', '/content/drive/MyDrive/Thesis/Data/teco_sentiment_labels_task1.csv',
           '--out-dir', out_dir, '--config', 'configs/v5_cnn_shuffle_control.json',
           '--reference-out-dir', reference_dir]
print('Starting matched-reference checks and CNN shuffle control:', flush=True)
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1) as process:
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        print('Stopped. Completed fits remain saved.', flush=True)
        raise
if returncode:
    raise subprocess.CalledProcessError(returncode, command)


## 3. Display and save the matched comparison
The two scores use different label mappings. Interpret their relationship descriptively; these two permutations do not provide a formal p-value.

In [ ]:
import pandas as pd
from IPython.display import display
control = pd.read_csv(os.path.join(out_dir, 'fold_metrics.csv'))
real = pd.read_csv(os.path.join(reference_dir, 'fold_metrics.csv'))
assert len(control) == len(real) == 24
assert not control[['seed', 'participant']].duplicated().any()
assert not real[['seed', 'participant']].duplicated().any()
fields = ['seed', 'participant', 'macro_f1', 'collapsed', 'learning_rate', 'validation_participant', 'fit_participants', 'n_fit', 'n_validation', 'n_test']
paired = real[fields].merge(control[fields], on=['seed', 'participant'], suffixes=('_real', '_shuffled'), validate='one_to_one')
assert len(paired) == 24
for name in ['learning_rate', 'validation_participant', 'fit_participants', 'n_fit', 'n_validation', 'n_test']:
    assert (paired[name + '_real'] == paired[name + '_shuffled']).all(), 'Mismatch: ' + name
for seed in [45, 62]:
    filename = f'shuffled_sentence_labels_seed_{seed}.csv'
    pd.testing.assert_frame_equal(pd.read_csv(os.path.join(out_dir, filename)),
                                  pd.read_csv(os.path.join(base, 'v5_fast_baselines_45_62', filename)))
summary = paired.groupby('seed', as_index=False).agg(
    participants=('participant', 'size'), real_macro_f1=('macro_f1_real', 'mean'),
    shuffled_macro_f1=('macro_f1_shuffled', 'mean'),
    real_collapses=('collapsed_real', 'sum'), shuffled_collapses=('collapsed_shuffled', 'sum'))
paired.to_csv(os.path.join(out_dir, 'matched_real_vs_shuffled.csv'), index=False)
summary.to_csv(os.path.join(out_dir, 'real_vs_shuffled_by_seed.csv'), index=False)
display(summary)
